# Установка зависимостей

In [1]:
!pip install pymorphy3
!pip install nltk
!pip install numpy==1.23.1
!pip install gensim==4.1.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 777.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 14.1 MB/s eta 0:00:00


In [3]:
import pandas as pd
import pymorphy3
import nltk
import pandas as pd
import gensim
import numpy as np
import re
nltk.download('stopwords')
nltk.download('punkt')
morph = pymorphy3.MorphAnalyzer()
from nltk.corpus import stopwords
from nltk.stem.snowball import RussianStemmer
from string import punctuation
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
from gensim.models import Word2Vec, FastText, KeyedVectors
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1.1 Создание базы данных

In [ ]:
path1 = '/content/drive/MyDrive/nlp/positive.csv'
path2 = '/content/drive/MyDrive/nlp/negative.csv'
data_positive = pd.read_csv(path1, delimiter = ';', header = None)
data_negative = pd.read_csv(path2, delimiter = ';', header = None)
shortened_data_positive = data_positive.sample(5000)
shortened_data_negative = data_negative.sample(5000)
df = pd.concat([shortened_data_negative, shortened_data_positive])

# Установить названия колонок
df.columns = ['id', 'date', 'name', 'text', 'type', 'rep', 'rtw', 'fav', 'stcount', 'foll', 'frien', 'listcount']
df=df.sample(10000)
df=df.reset_index(drop=True)

In [ ]:
df.head()

,id,date,name,text,type,rep,rtw,fav,stcount,foll,frien,listcount
0,417648589956214784,1388410158,ksyuschenkakony,"Вчера играла на гитаре« реквием по мечте»,пале...",-1,0,0,0,62,3,16,0
1,413675603011727361,1387462924,StarkAurora,"RT @30SecondsAl: @StarkAurora ооо,чувак :(",-1,0,1,0,6948,1360,1352,2
2,418620526232616960,1388641885,ZloyTwit,"Бля не доехал до кожевниковой, уснул( сейчас с...",-1,0,0,0,21648,262,108,2
3,409756065404706816,1386528433,NsPani,"@DIRICFOREVER сладких снов,малыыш:*дадад,поско...",1,0,0,0,17596,2136,2054,16
4,410064640392318977,1386602003,Yrman2Yrman,"RT @TukvaSociopat: ""@lazarev_ua: На Майдане мн...",1,0,45,0,51821,188,131,7


# 1.2 Предобработка текста

In [7]:
# Загружаем стоп-слова и убираем "не"
stopwords = set(nltk.corpus.stopwords.words('russian')) - {'не'}

# Инициализируем лемматизатор
morph = pymorphy3.MorphAnalyzer()

# Функция предобработки текста
def preprocess_text(text):
    text = re.sub('[^А-Яа-яйЙёЁ]', ' ', text)  # Удаление лишних символов
    tokens = re.findall(r'\b\w+\b', text.lower())  # Токенизация и приведение к нижнему регистру
    tokens = [word for word in tokens if word not in punctuation and word not in stopwords]  # Удаление пунктуации и стоп-слов
    lemmas = [morph.parse(word)[0].normal_form for word in tokens]  # Лемматизация
    return ' '.join(lemmas)  # Объединение в строку

# Применяем предобработку ко всем текстам
clean_df = df[['text']].copy()  # Создаем копию датафрейма с нужным столбцом
clean_df['clean_text'] = clean_df['text'].apply(preprocess_text)

# Сохраняем в CSV
clean_df.to_csv('/content/drive/MyDrive/nlp/clean_df.csv', index=False)

# Выводим первые строки
clean_df.head()

NameError: name 'df' is not defined

# 1.3 Тональный словарь LinisCrowd 2015

In [ ]:
sentiment_dict = pd.read_csv('/content/drive/MyDrive/nlp/words_all_full_rating.csv', encoding='windows-1251', delimiter=';')
sentiment_dict['mean'] = sentiment_dict['mean'].str.replace(',', '.').astype(float)
sentiment_dict = dict(zip(sentiment_dict['Words'], sentiment_dict['mean']))

clean_df['liniscrowd'] = clean_df['clean_text'].apply(lambda x: [round(sentiment_dict.get(x_split, 0), 2) for x_split in str(x).split(' ')])

In [ ]:
def liniscrowd_values(sentiment_scores):
  results = []
  if not sentiment_scores:
    sentiment_scores = [0]

  results.append([
      sum(sentiment_scores) / len(sentiment_scores),  # Среднее
      max(sentiment_scores),  # Максимальное
      min(sentiment_scores),  # Минимальное
      sum(sentiment_scores),  # Сумма
      sum(1 for s in sentiment_scores if s > 0),  # Количество положительных
      sum(1 for s in sentiment_scores if s < 0)   # Количество отрицательных
      ])

  return results

In [ ]:
clean_df['liniscrowd_values'] = clean_df['liniscrowd'].apply(liniscrowd_values)

In [ ]:
clean_df

,clean_text,liniscrowd,liniscrowd_values
0,вчера играть гитара реквием мечта палец левый ...,"[0, 0.0, 0, 0, 0.67, 0, 0.0, 0, 0, 0]","[[0.067, 0.67, 0, 0.67, 1, 0]]"
1,ооо чувак,"[0, 0]","[[0.0, 0, 0, 0, 0, 0]]"
2,бля не доехать кожевников уснуть скоро попрать,"[0, 0, 0, 0, 0.0, 0, 0]","[[0.0, 0, 0, 0.0, 0, 0]]"
3,сладкий сон малыыш дадада скорый мы ильгиз ещё...,"[0.89, 0, 0, 0, 0, 0, 0, 0, 0.0, 0, 0]","[[0.08090909090909092, 0.89, 0, 0.89, 1, 0]]"
4,майдан человек вромайдан,"[0, 0, 0]","[[0.0, 0, 0, 0, 0, 0]]"
...,...,...,...
9995,наркота это зелёный чай прокатить хула,"[-1.67, 0, 0, 0, 0, 0]","[[-0.2783333333333333, 0, -1.67, -1.67, 0, 1]]"
9996,завтра поехать час пара последний день выходной,"[0, 0, 0.0, 0, 0, 0, 0]","[[0.0, 0, 0, 0.0, 0, 0]]"
9997,хотеть ранний не,"[0.0, 0.0, 0]","[[0.0, 0.0, 0.0, 0.0, 0, 0]]"
9998,сочувствовать это страшно именно поэтому стара...,"[1.0, 0, 0, 0, 0, 0, 0, 0, 0, -1.33, 0, 0, 0, 0]","[[-0.023571428571428577, 1.0, -1.33, -0.330000..."


# 1.4 PoS

In [ ]:
def get_pos_frequencies(text):
    words = text.split()
    pos_counts = Counter()

    for word in words:
        parsed_word = morph.parse(word)[0]
        pos_counts[parsed_word.tag.POS] += 1

    total_words = sum(pos_counts.values())
    if total_words > 0:
        pos_freq = {pos: count / total_words for pos, count in pos_counts.items()}
    else:
        pos_freq = {}
    return pos_freq

In [ ]:
clean_df['pos_frequencies'] = clean_df['clean_text'].apply(get_pos_frequencies)
df_pos = clean_df['pos_frequencies'].apply(pd.Series).fillna(0)
clean_df = pd.concat([clean_df, df_pos], axis=1).drop(columns=['pos_frequencies'])

In [ ]:
texts = clean_df['clean_text'].fillna('')

In [ ]:
clean_df.head(n=10)

,clean_text,liniscrowd,liniscrowd_values,ADVB,INFN,NOUN,ADJF,INTJ,PRCL,NPRO,...,CONJ,None,PRED,VERB,ADJS,PREP,GRND,PRTF,PRTS,COMP
0,вчера играть гитара реквием мечта палец левый ...,"[0, 0.0, 0, 0, 0.67, 0, 0.0, 0, 0, 0]","[[0.067, 0.67, 0, 0.67, 1, 0]]",0.100000,0.200000,0.600000,0.100000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ооо чувак,"[0, 0]","[[0.0, 0, 0, 0, 0, 0]]",0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,бля не доехать кожевников уснуть скоро попрать,"[0, 0, 0, 0, 0.0, 0, 0]","[[0.0, 0, 0, 0.0, 0, 0]]",0.142857,0.428571,0.142857,0.000000,0.142857,0.142857,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,сладкий сон малыыш дадада скорый мы ильгиз ещё...,"[0.89, 0, 0, 0, 0, 0, 0, 0, 0.0, 0, 0]","[[0.08090909090909092, 0.89, 0, 0.89, 1, 0]]",0.090909,0.181818,0.454545,0.181818,0.000000,0.000000,0.090909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,майдан человек вромайдан,"[0, 0, 0]","[[0.0, 0, 0, 0, 0, 0]]",0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,ахи согласный зомби апокалипсис,"[0, 0.67, -1.11, 0]","[[-0.11000000000000001, 0.67, -1.11, -0.440000...",0.000000,0.000000,0.750000,0.250000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,стыдно результат тест знание русский язык семь...,"[-1.67, 0.0, 0, 0, 0, -0.33, 0, 0.4, 0, 0]","[[-0.16, 0.4, -1.67, -1.6, 1, 2]]",0.100000,0.000000,0.500000,0.200000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,пойти отдохнуть минута проспать час,"[0.0, 0, 0, 0, 0.0]","[[0.0, 0.0, 0.0, 0.0, 0, 0]]",0.000000,0.600000,0.400000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,очень долго вспоминать звать новый секретарша ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0]","[[0.0, 0, 0, 0, 0, 0]]",0.222222,0.333333,0.222222,0.111111,0.000000,0.111111,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,добрый задерживаться,"[1.33, 0]","[[0.665, 1.33, 0, 1.33, 1, 0]]",0.000000,0.500000,0.000000,0.500000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 1.5 TF-IDF

In [ ]:
def extract_tfidf_features(texts, max_features=1000):
    vectorizer = TfidfVectorizer(max_features=max_features)
    tfidf_matrix = vectorizer.fit_transform(texts)
    return tfidf_matrix.toarray()

In [ ]:
tfidf_features = extract_tfidf_features(clean_df["clean_text"])

In [ ]:
# Преобразуем tfidf_features в DataFrame
tfidf_df = pd.DataFrame(tfidf_features)

# Сохраняем DataFrame в CSV
tfidf_df.to_csv('tfidf_features.csv', index=False)

In [ ]:
tfidf_df.head(n=25)

,0,1,2,3,4,5,6,7,8,9,...,990,991,992,993,994,995,996,997,998,999
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.46756,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0


# Wav2Vec

In [5]:
path1 = '/content/drive/MyDrive/nlp/positive.csv'
path2 = '/content/drive/MyDrive/nlp/negative.csv'
data_positive = pd.read_csv(path1, delimiter = ';', header = None)
data_negative = pd.read_csv(path2, delimiter = ';', header = None)
full_df = pd.concat([data_negative, data_positive])

# Установить названия колонок
full_df.columns = ['id', 'date', 'name', 'text', 'type', 'rep', 'rtw', 'fav', 'stcount', 'foll', 'frien', 'listcount']
full_df=full_df.reset_index(drop=True)

In [8]:
# Применяем предобработку ко всем текстам
full_clean_df = full_df[['type', 'text']].copy()
full_clean_df['clean_text'] = full_clean_df['text'].apply(preprocess_text)
full_clean_df.drop(columns=['text'], inplace=True)

# Сохраняем в CSV
full_clean_df.to_csv('/content/drive/MyDrive/nlp/full_clean_df.csv', index=False)

# Выводим первые строки
full_clean_df.head()

,type,clean_text
0,-1,работа полный пиддес каждый закрытие месяц сви...
1,-1,коллега сидеть рубиться долбать винд не мочь
2,-1,говорить обещаной год ждать
3,-1,желать хороший полёт удачный посадка быть очен...
4,-1,обновить какой леший не работать простоплеер


In [17]:
# Преобразуем текст в список слов для каждого предложения
sentences = [text.split() for text in full_clean_df['clean_text']]

# Создаем модель Word2Vec
w2vmodel = Word2Vec(sentences, vector_size=300, window=5, min_count=1, workers=4)

# Попробуем найти похожие слова
w2vmodel.wv.most_similar('хотеть')

[('хотеться', 0.8501813411712646),
 ('захотеть', 0.7652725577354431),
 ('охота', 0.7450435757637024),
 ('двуспальный', 0.7423616051673889),
 ('пилон', 0.7393417954444885),
 ('охоть', 0.73207026720047),
 ('задроток', 0.731861412525177),
 ('захотеться', 0.7300191521644592),
 ('нужно', 0.7260615229606628),
 ('хочеться', 0.7177619934082031)]

In [18]:
print(list(w2vmodel.wv.index_to_key))

['не', 'это', 'хотеть', 'день', 'такой', 'сегодня', 'быть', 'очень', 'мой', 'ты', 'знать', 'год', 'просто', 'мочь', 'человек', 'весь', 'любить', 'завтра', 'новый', 'вообще', 'свой', 'всё', 'делать', 'один', 'хороший', 'спасибо', 'смотреть', 'блин', 'почему', 'сказать', 'думать', 'который', 'время', 'идти', 'кто', 'говорить', 'спать', 'дом', 'самый', 'друг', 'сидеть', 'пойти', 'утро', 'писать', 'жизнь', 'сделать', 'школа', 'настроение', 'ждать', 'сам', 'первый', 'мама', 'какой', 'час', 'тот', 'ещё', 'наш', 'болеть', 'любимый', 'твой', 'написать', 'хотеться', 'работа', 'видеть', 'ночь', 'посмотреть', 'пока', 'никто', 'прийти', 'понять', 'последний', 'стать', 'добрый', 'х', 'дело', 'купить', 'понимать', 'неделя', 'мы', 'найти', 'давать', 'этот', 'каждый', 'нравиться', 'скучать', 'скоро', 'нужно', 'урок', 'вчера', 'забыть', 'читать', 'снег', 'плохо', 'работать', 'фильм', 'голова', 'остаться', 'хотя', 'жить', 'вечер', 'ходить', 'надеяться', 'домой', 'ахи', 'пара', 'телефон', 'жаль', 'сон', 

In [24]:
for sent in full_clean_df['clean_text']:  # Заменим на правильный доступ
    sent_words = sent.split()  # Разделим предложение на слова
    vectors = []
    for word in sent_words:
        try:
            vectors.append(w2vmodel.wv[word])  # Теперь это слово, а не символ
        except KeyError:
            vectors.append(np.zeros(300))  # Если слова нет в модели, добавим нулевой вектор
    if len(vectors) == 0:
       w2v['vector'].append(np.zeros(300))
    else:
       w2v['vector'].append(np.array(np.mean(vectors, axis=0)))

In [26]:
w2v_df = pd.concat([full_clean_df[['type']], pd.DataFrame(w2v['vector'])], axis=1)
w2v_df.to_csv(r'/content/drive/MyDrive/nlp/w2v.csv', index=False)
w2v_df

,type,0,1,2,3,4,5,6,7,8,...,290,291,292,293,294,295,296,297,298,299
0,-1,0.297804,0.186753,-0.356145,-0.123329,-0.181119,-0.466399,0.219948,0.426470,-0.402852,...,-0.005466,0.606232,0.392969,0.028447,0.394043,0.238019,-0.134320,-0.128036,0.024439,-0.419876
1,-1,0.185061,0.293969,-0.068373,0.009485,-0.312729,-0.511153,0.337169,0.368253,-0.189733,...,-0.030654,0.503372,0.507958,-0.040097,0.349439,0.353044,0.150932,-0.061326,0.354985,-0.429854
2,-1,0.346427,0.485325,-0.358036,-0.200172,-0.001445,-0.558185,0.584730,0.987722,-0.202034,...,0.022045,0.934773,0.667638,-0.054105,0.402328,0.338382,0.226691,0.166161,0.131215,-0.644639
3,-1,0.205367,0.301637,-0.108057,-0.054055,-0.170576,-0.388915,0.344160,0.851197,-0.432621,...,0.050428,0.509660,0.222069,0.032953,0.420633,0.638506,0.249868,0.081213,-0.018765,-0.673834
4,-1,-0.060083,0.109679,0.007216,-0.175025,-0.316350,-0.378468,0.280697,0.372613,0.006992,...,0.174423,0.685863,0.519252,-0.086315,0.438004,0.187381,0.004280,-0.420728,0.430668,-0.311877
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,1,0.357280,0.214511,-0.396161,-0.145322,-0.368856,-0.456776,0.056215,0.607541,-0.435988,...,0.116558,0.607872,0.663979,-0.015583,0.871310,0.530141,0.336487,-0.133498,0.458751,-0.216855
226830,1,0.120864,0.264321,-0.118672,0.101784,-0.234991,-0.393908,0.198978,0.504070,-0.167727,...,0.074675,0.633812,0.454746,-0.119994,0.440336,0.437144,0.081421,-0.132695,0.392951,-0.256368
226831,1,0.245320,0.361384,-0.159219,-0.241291,-0.179979,-0.483108,0.625755,0.510336,-0.320781,...,0.125147,0.652197,0.407174,-0.019753,0.614020,0.208148,0.335952,-0.074379,0.257992,-0.567258
226832,1,0.056434,0.196153,0.156358,0.099600,-0.021514,-0.236631,0.533325,0.584916,0.051866,...,0.009906,0.558378,0.339919,-0.123536,0.321504,0.312732,0.192249,-0.094829,0.291869,-0.205717


In [39]:
#model_path = "ruwikiruscorpora_upos_cbow_300_10_2021.bin.gz"
w2vmodel_pretrained = gensim.models.KeyedVectors.load_word2vec_format('/content/drive/MyDrive/nlp/model.bin', encoding='utf-8', unicode_errors='ignore', binary=True)
w2vmodel_pretrained.most_similar('красивый_ADJ')

[('прекрасный_ADJ', 0.7402893900871277),
 ('элегантный_ADJ', 0.7000579237937927),
 ('прелестный_ADJ', 0.6995062828063965),
 ('привлекательный_ADJ', 0.670931339263916),
 ('очаровательный_ADJ', 0.6690368056297302),
 ('некрасивый_ADJ', 0.6563753485679626),
 ('великолепный_ADJ', 0.6439347863197327),
 ('симпатичный_ADJ', 0.6420075297355652),
 ('нарядный_ADJ', 0.6389022469520569),
 ('стильный_ADJ', 0.6302294731140137)]

In [40]:
w2vmodel_cleaned = {key.split('_')[0]: w2vmodel_pretrained[key] for key in w2vmodel_pretrained.key_to_index}

In [41]:
p = morph.parse('печь')[0]
first_tag_part = p.tag.POS
first_tag_part

'NOUN'

In [42]:
w2v_pretrained = {'text': [], 'vector': []}

for sent in full_clean_df['clean_text']:  # Извлекаем текст из DataFrame
  w2v_pretrained['text'].append(sent)  # Добавляем текст в список

  vectors = []
  words = sent.split()  # Разбиваем предложение на слова

  for word in words:  # Для каждого слова
    try:
      vectors.append(w2vmodel_cleaned[word])  # Пытаемся получить вектор
    except KeyError:
      vectors.append(np.zeros(300))  # Если слова нет в модели, добавляем нулевой вектор

  # Если векторов нет, добавляем нулевой вектор для всего предложения
  if len(vectors) == 0:
    w2v_pretrained['vector'].append(np.zeros(300))
  else:
    # Добавляем средний вектор для предложения
    w2v_pretrained['vector'].append(np.array(np.mean(vectors, axis=0)))

# Теперь можно создать DataFrame из векторов
w2v_df = pd.DataFrame(w2v_pretrained['vector'])

In [43]:
w2v_pretrained_df = pd.concat([full_clean_df[['type']],  pd.DataFrame(w2v_pretrained['vector'])], axis=1 )
w2v_pretrained_df.to_csv(r'/content/drive/MyDrive/nlp/w2v_pretrained.csv', index=False)
w2v_pretrained_df

,type,0,1,2,3,4,5,6,7,8,...,290,291,292,293,294,295,296,297,298,299
0,-1,-0.279118,-0.311169,0.354438,0.153770,0.197746,-0.339183,0.338439,-0.216828,0.220528,...,-0.535218,-0.243805,-0.216478,0.442656,0.212496,0.470391,0.077077,-0.200527,0.086078,0.398163
1,-1,-0.029762,0.140600,0.133712,0.035470,0.075019,0.000115,0.020909,-0.183372,-0.129579,...,0.001369,-0.056840,0.095804,0.272807,-0.103780,-0.115116,0.066937,-0.071950,-0.190474,0.074695
2,-1,0.306034,-0.049628,0.511388,-0.133388,0.627173,0.016263,0.544889,-0.487313,0.000532,...,0.036866,-0.223869,-0.066166,0.483355,0.093465,-0.020690,0.107873,-0.225651,-0.253817,0.056730
3,-1,0.189057,-0.725490,0.466915,0.365590,-0.165439,-0.388688,0.304931,0.133061,-0.219910,...,0.306416,0.209425,-0.203804,0.852882,-0.137055,-0.092770,0.046888,0.088987,-0.234186,-0.041974
4,-1,-0.410390,-0.518762,-0.444348,0.288997,-0.094432,-0.220770,-0.299935,0.184940,-0.000381,...,-0.432770,0.057735,-0.012028,-0.013468,-0.057108,0.544200,-0.348487,0.289541,-0.089091,-0.668194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,1,-0.825148,-0.744613,-0.097037,-0.130733,0.342291,-0.312562,-0.105434,0.087703,0.185391,...,0.056417,0.213189,0.015083,0.104531,0.737710,0.004642,0.449725,0.146682,-1.306366,0.327593
226830,1,-0.040685,0.543406,0.377491,0.158974,0.284649,0.254869,0.498592,-0.407123,-0.465529,...,0.058191,0.089812,-0.221591,-0.894840,-0.096231,0.406389,-0.168242,0.338954,-0.175192,0.738967
226831,1,0.531125,-0.659506,0.492416,0.164096,0.120235,-0.463958,0.549875,0.139749,1.109016,...,-0.191787,0.111274,-0.558082,0.467569,0.051947,0.142904,0.005607,-0.626344,0.119670,-0.071015
226832,1,0.060531,-0.103418,-0.188757,0.214896,-0.007337,-0.275111,-0.249263,0.630395,0.429653,...,-0.617296,0.220835,0.014478,0.272023,-0.210607,-0.002705,0.223202,-0.585158,-0.472897,-0.013062


# FastText

In [45]:
# Разделяем текст на слова
sentences = [text.split() for text in full_clean_df['clean_text']]

# Обучаем модель FastText
ftmodel = FastText(sentences, vector_size=300, window=5, min_count=5)

# Получаем похожие слова
print(ftmodel.wv.most_similar('плохо'))

[('плохоо', 0.9758578538894653), ('плоохо', 0.9339907169342041), ('плоховато', 0.9220544695854187), ('неплохо', 0.9144150018692017), ('охохо', 0.9022649526596069), ('охохохо', 0.9016557931900024), ('хохохо', 0.8450205326080322), ('нудно', 0.8431478142738342), ('плодотворно', 0.83534175157547), ('ясно', 0.8343384861946106)]


In [57]:
print(ftmodel.wv.most_similar('хотеть'))

[('нехотеть', 0.9557750225067139), ('захотеть', 0.9103893041610718), ('перехотеть', 0.895476758480072), ('хотеться', 0.8819913268089294), ('захотеться', 0.844386637210846), ('расхотеться', 0.8235397934913635), ('перехотеться', 0.8136854767799377), ('потеть', 0.7846387624740601), ('хотяй', 0.7225399613380432), ('хотяб', 0.7188836336135864)]


In [54]:
fasttext = {'text': [], 'vector': []}

for sent in full_clean_df['clean_text']:  # Предполагается, что ваши предложения в столбце 'clean_text'
    fasttext['text'].append(sent)
    vectors = []

    # Разбиваем предложение на слова
    words = sent.split()

    for word in words:
        try:
            # Получаем вектор слова из модели FastText
            vectors.append(ftmodel.wv[word])
        except KeyError:
            # Если слова нет в модели, добавляем вектор из нулей
            vectors.append(np.zeros(300))

    # Если нет векторов, добавляем нулевой вектор
    if len(vectors) == 0:
        fasttext['vector'].append(np.zeros(300))
    else:
        # Усредняем векторы слов для всего предложения
        fasttext['vector'].append(np.array(np.mean(vectors, axis=0)))

# Создаем DataFrame с текстами и векторами
fasttext_df = pd.DataFrame(fasttext)

# Теперь сохраняем DataFrame
fasttext_df.to_csv(r'/content/drive/MyDrive/nlp/fasttext.csv', index=False)

# Выводим первые 5 строк DataFrame для проверки
print(fasttext_df.head())

                                                text  \
0  работа полный пиддес каждый закрытие месяц сви...   
1       коллега сидеть рубиться долбать винд не мочь   
2                        говорить обещаной год ждать   
3  желать хороший полёт удачный посадка быть очен...   
4       обновить какой леший не работать простоплеер   

                                              vector  
0  [0.12392359, -0.090507455, -0.2566858, -0.3712...  
1  [0.26639965, -0.03203414, -0.37419733, -0.3410...  
2  [0.008327104, 0.2917432, 0.13379449, -0.564460...  
3  [-0.14760642, 0.0807944, -0.16047563, -0.25633...  
4  [0.4231386, -0.030744314, -0.0892599, -0.29047...  


In [55]:
# Создаем DataFrame для векторов FastText
fasttext_df = pd.DataFrame(fasttext['vector'])

# Проверяем размеры
print(fasttext_df.shape)  # Убедимся, что количество строк совпадает с full_clean_df

# Объединяем DataFrame с типами и векторами
fasttext_df = pd.concat([full_clean_df[['type']], fasttext_df], axis=1)

# Сохраняем итоговый DataFrame в файл
fasttext_df.to_csv(r'/content/drive/MyDrive/nlp/fasttext_with_type.csv', index=False)

# Выводим первые 5 строк для проверки
print(fasttext_df.head())

(226834, 300)
   type         0         1         2         3         4         5         6  \
0    -1  0.123924 -0.090507 -0.256686 -0.371285  0.084424 -0.303838  0.037916   
1    -1  0.266400 -0.032034 -0.374197 -0.341049  0.329782 -0.122250  0.052684   
2    -1  0.008327  0.291743  0.133794 -0.564461 -0.061580 -0.355557 -0.268032   
3    -1 -0.147606  0.080794 -0.160476 -0.256338  0.115687 -0.321475  0.191327   
4    -1  0.423139 -0.030744 -0.089260 -0.290471  0.243709 -0.344473  0.590232   

          7         8  ...       290       291       292       293       294  \
0  0.107319  0.304985  ... -0.152093 -0.329242 -0.495600 -0.067512 -0.225804   
1 -0.087462 -0.212627  ...  0.416603 -0.349920 -0.236766  0.263482  0.094216   
2  0.358045  0.280615  ... -0.516658 -0.170752 -0.487127  0.288966 -0.223147   
3  0.080355  0.093295  ...  0.202840 -0.180542 -0.650065  0.235518 -0.204893   
4 -0.057174  0.076529  ...  0.575124 -0.149423 -0.238259 -0.129575 -0.234343   

        295       

In [9]:
ftmodel_pretrained = gensim.models.KeyedVectors.load('/content/drive/MyDrive/nlp/model.model')
ftmodel_pretrained.most_similar('красивый')

[(',красивый', 0.8366382718086243),
 ('.красивый', 0.8338299989700317),
 ('красивенный', 0.7265939116477966),
 ('красивенький', 0.7174873352050781),
 ('симпатичный', 0.7136726975440979),
 ('шикарный', 0.7124730348587036),
 ('эффектный', 0.70670485496521),
 ('красочный', 0.6997988820075989),
 ('очаровательный', 0.6937825679779053),
 ('потрясающий', 0.6877936124801636)]

In [71]:
ft_pretrained = {'text': [], 'vector': []}
for sent in full_clean_df:
  ft_pretrained['text'].append(sent)
  vectors = []
  for word in sent:
    vectors.append(ftmodel_pretrained[word])
  if len(vectors) == 0:
    ft_pretrained['vector'].append(np.zeros(300))
  else:
    ft_pretrained['vector'].append(np.array(np.mean(vectors, axis=0)))

In [5]:
full_clean_df = pd.read_csv('/content/drive/MyDrive/nlp/full_clean_df.csv')

In [13]:
ft_pretrained = {'text': [], 'vector': []}

for sent in full_clean_df['clean_text']:  # используем столбец с текстом
    if isinstance(sent, str):  # проверяем, что это строка
        ft_pretrained['text'].append(sent)
        vectors = []
        for word in sent.split():  # Разбиваем текст на слова
            try:
                vectors.append(ftmodel_pretrained[word])
            except KeyError:
                vectors.append(np.zeros(300))  # Если слова нет в модели, добавляем вектор из нулей

        # Если нет векторов, добавляем нулевой вектор
        if len(vectors) == 0:
            ft_pretrained['vector'].append(np.zeros(300))
        else:
            ft_pretrained['vector'].append(np.array(np.mean(vectors, axis=0)))
    else:
        # Если значение не строка (например, NaN или число), добавляем пустой вектор
        ft_pretrained['text'].append('')
        ft_pretrained['vector'].append(np.zeros(300))

# Создаем DataFrame с векторами
ft_pretrained_df = pd.DataFrame(ft_pretrained['vector'])

In [14]:
fasttext_pretrainef_df = pd.concat([full_clean_df[['type']],  pd.DataFrame(ft_pretrained['vector'])], axis=1 )
fasttext_pretrainef_df.to_csv(r'/content/drive/MyDrive/nlp/fasttext_pretr.csv', index=False)
fasttext_pretrainef_df

,type,0,1,2,3,4,5,6,7,8,...,290,291,292,293,294,295,296,297,298,299
0,-1,-0.020151,0.069581,-0.053569,0.247671,0.137248,0.200066,-0.015692,0.134414,0.054312,...,0.026138,0.147686,0.087791,-0.046578,-0.166233,0.080708,0.068734,-0.109995,-0.293939,-0.004935
1,-1,0.122010,0.077201,-0.020517,0.172850,0.070921,-0.009873,0.103921,-0.023100,0.101336,...,0.203994,0.113248,0.111351,0.021807,-0.182379,0.034013,0.046017,-0.062362,-0.182148,-0.060355
2,-1,-0.063084,-0.015199,-0.170328,0.011000,0.212626,0.262083,-0.008568,0.071123,-0.030609,...,0.061179,-0.031331,0.097060,0.142485,-0.079269,-0.090395,0.149548,0.015660,-0.199339,0.008175
3,-1,-0.010186,0.023974,-0.074630,0.426821,0.208599,0.056377,-0.021979,-0.062656,0.264763,...,-0.016054,0.097827,0.163598,0.069814,-0.139596,0.031154,0.137533,-0.106403,-0.108858,0.115289
4,-1,0.100567,0.100108,-0.069529,0.122550,0.153471,0.027437,0.008886,0.001047,0.042476,...,0.093093,0.146769,0.108994,0.075767,-0.106071,0.004924,0.222818,-0.144029,-0.187789,-0.014031
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,1,0.063053,0.068928,-0.038545,0.062647,0.004820,0.013285,-0.021656,-0.076963,-0.026101,...,0.001430,0.073476,0.090030,0.110430,-0.095829,-0.042353,0.067769,-0.106041,-0.222030,-0.058689
226830,1,0.041691,-0.022612,-0.079342,0.170320,0.066096,-0.069708,0.080173,0.060388,0.070476,...,0.016562,0.127788,0.085547,0.052909,-0.036571,0.078502,0.169290,-0.105821,-0.038792,0.004138
226831,1,0.159563,0.102877,-0.171595,0.015761,0.218519,0.030864,-0.085978,-0.219051,-0.082539,...,-0.045249,0.115453,0.105802,0.081790,-0.128821,-0.039213,0.101940,0.030473,-0.018238,-0.031799
226832,1,-0.016787,0.037712,-0.086821,0.125392,0.127242,0.139783,-0.014138,-0.219812,-0.006541,...,-0.048324,0.192771,0.209858,0.000350,-0.178893,0.001481,-0.098127,0.012529,-0.170353,0.062385


# Обучение

In [23]:
results_lr = {}

results_rf = {}

def get_vectors(path):
  data = pd.read_csv(path)
  features = data.drop('type', axis=1)
  labels = data['type']
  return features, labels

def train(features, lables, model, data_type):
  X_train, X_test, y_train, y_test = train_test_split(features, lables, test_size=0.2, random_state=42)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  if model == lr_model:
    results_lr[data_type] = accuracy_score(y_test, y_pred)
  if model == rf_model:
    results_rf[data_type] = accuracy_score(y_test, y_pred)
  print(classification_report(y_test, y_pred))

In [24]:
lr_model = LogisticRegression()
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

In [25]:
wav2vec_features, wav2vec_labels = get_vectors(r'/content/drive/MyDrive/nlp/w2v.csv')
train(wav2vec_features, wav2vec_labels, lr_model, 'wav2vec')
train(wav2vec_features, wav2vec_labels, rf_model, 'wav2vec')

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

          -1       0.64      0.60      0.62     22298
           1       0.64      0.68      0.66     23069

    accuracy                           0.64     45367
   macro avg       0.64      0.64      0.64     45367
weighted avg       0.64      0.64      0.64     45367

              precision    recall  f1-score   support

          -1       0.68      0.67      0.67     22298
           1       0.68      0.69      0.69     23069

    accuracy                           0.68     45367
   macro avg       0.68      0.68      0.68     45367
weighted avg       0.68      0.68      0.68     45367



In [26]:
wav2vec_pretr_features, wav2vec_pretr_labels = get_vectors(r'/content/drive/MyDrive/nlp/w2v_pretrained.csv')
train(wav2vec_pretr_features, wav2vec_pretr_labels, lr_model, 'wav2vec_pretrained')
train(wav2vec_pretr_features, wav2vec_pretr_labels, rf_model, 'wav2vec_pretrained')

              precision    recall  f1-score   support

          -1       0.65      0.62      0.63     22298
           1       0.65      0.68      0.66     23069

    accuracy                           0.65     45367
   macro avg       0.65      0.65      0.65     45367
weighted avg       0.65      0.65      0.65     45367

              precision    recall  f1-score   support

          -1       0.66      0.64      0.65     22298
           1       0.66      0.68      0.67     23069

    accuracy                           0.66     45367
   macro avg       0.66      0.66      0.66     45367
weighted avg       0.66      0.66      0.66     45367



In [30]:
ft_features, ft_labels= get_vectors(r'/content/drive/MyDrive/nlp/fasttext_with_type.csv')
train(ft_features, ft_labels, lr_model, 'fast text')
train(ft_features, ft_labels, rf_model, 'fast text')

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

          -1       0.65      0.60      0.62     22298
           1       0.64      0.69      0.66     23069

    accuracy                           0.64     45367
   macro avg       0.64      0.64      0.64     45367
weighted avg       0.64      0.64      0.64     45367

              precision    recall  f1-score   support

          -1       0.67      0.67      0.67     22298
           1       0.68      0.69      0.68     23069

    accuracy                           0.68     45367
   macro avg       0.68      0.68      0.68     45367
weighted avg       0.68      0.68      0.68     45367



In [31]:
ft_pretr_features, ft_pretr_labels  = get_vectors(r'/content/drive/MyDrive/nlp/fasttext_pretr.csv')
train(ft_pretr_features, ft_pretr_labels, lr_model, 'fast text pretrained')
train(ft_pretr_features, ft_pretr_labels , rf_model, 'fast text pretrained')

              precision    recall  f1-score   support

          -1       0.68      0.66      0.67     22298
           1       0.68      0.70      0.69     23069

    accuracy                           0.68     45367
   macro avg       0.68      0.68      0.68     45367
weighted avg       0.68      0.68      0.68     45367

              precision    recall  f1-score   support

          -1       0.69      0.69      0.69     22298
           1       0.70      0.70      0.70     23069

    accuracy                           0.69     45367
   macro avg       0.69      0.69      0.69     45367
weighted avg       0.69      0.69      0.69     45367



# Результаты классификации с помощью логистической регрессии

In [32]:
results_lr_df = pd.DataFrame.from_dict(results_lr, orient='index', columns=['Accuracy'])
results_lr_df

,Accuracy
wav2vec,0.639341
wav2vec_pretrained,0.649569
fast text,0.644433
fast text pretrained,0.683184


In [33]:
results_lr_df = pd.DataFrame.from_dict(results_lr, orient='index', columns=['Accuracy'])
results_lr_df

,Accuracy
wav2vec,0.639341
wav2vec_pretrained,0.649569
fast text,0.644433
fast text pretrained,0.683184


# Результаты классификации с помощью метода случайного леса

In [34]:
results_svc_df = pd.DataFrame.from_dict(results_rf, orient='index', columns=['Accuracy'])
results_svc_df

,Accuracy
wav2vec,0.678224
wav2vec_pretrained,0.661604
fast text,0.677739
fast text pretrained,0.692949


In [35]:
results_svc_df = pd.DataFrame.from_dict(results_rf, orient='index', columns=['Accuracy'])
results_svc_df

,Accuracy
wav2vec,0.678224
wav2vec_pretrained,0.661604
fast text,0.677739
fast text pretrained,0.692949
